# 12.8 · 预训练 vs 微调 / Pretraining vs Fine-tuning

> **课程定位 / Where this fits**
> 第 8 课，**Part 12**。理解 LLM 怎么"造出来 + 用起来"的核心范式。
> Lesson 8, **Part 12**. The core paradigm of how LLMs are "built and adapted."
>
> 现代 LLM 走**两阶段**：**预训练(pretraining)** ——在**海量无标注文本**上用"预测下一个 token"自监督训练，学到通用的语言能力与世界知识(极贵, 几百万美元、几千张 GPU, 只有少数机构能做)；**微调(fine-tuning)** ——在预训练模型上用**少量特定数据**适配到具体任务/领域/风格(便宜, 人人可做)。这套"通用预训练 + 任务微调"是 LLM 时代最重要的工程范式。本课讲清两者的概念、区别、**实测微调 vs 从零**，以及"何时该微调、何时用别的方法"。
> Modern LLMs follow **two stages**: **pretraining** — self-supervised "predict the next token" on **massive unlabeled text**, learning general language ability and world knowledge (very expensive — millions of dollars, thousands of GPUs, few can do it); **fine-tuning** — adapting the pretrained model with **little task-specific data** to a task/domain/style (cheap, anyone can do). This "general pretrain + task fine-tune" is the most important engineering paradigm of the LLM era. We clarify both, **measure fine-tune vs from-scratch**, and discuss "when to fine-tune vs use other methods."
>
> 💼 **实战/面试视角**："预训练学到什么 / 微调何时用 / 全量微调 vs PEFT vs prompt vs RAG 怎么选 / 灾难性遗忘" 是落地 LLM 必问。
> 💼 **Practical/interview angle:** "what pretraining learns / when to fine-tune / full fine-tune vs PEFT vs prompt vs RAG / catastrophic forgetting" — must-knows for shipping LLMs.

> 💡 **面试相关 / Interview-relevant**
> - "预训练 vs 微调的区别与各自成本"（出镜率 ★★★★★）
> - "什么时候该微调, 什么时候用 prompt/RAG"（★★★★★）
> - "指令微调(instruction tuning)是什么"（★★★★）
> - "灾难性遗忘"（★★★）

---

## 学习目标 / Learning Objectives
1. 理解预训练→微调两阶段范式及成本差异。
   Understand the pretrain→fine-tune paradigm and cost asymmetry.
2. 理解预训练(自监督)到底学到了什么。
   Understand what (self-supervised) pretraining learns.
3. **实测**微调预训练模型 vs 从零训练。
   Measure fine-tuning a pretrained model vs training from scratch.
4. 学会决策: 微调 / PEFT / prompt / RAG 何时用。
   Decide when to use fine-tuning / PEFT / prompting / RAG.

## 目录 / TOC
1. [两阶段范式 ⭐](#1)
2. [预训练学到了什么 ⭐](#2)
3. [实测：微调 vs 从零 ⭐](#3)
4. [决策指南: 微调/PEFT/Prompt/RAG + 小结 ⭐](#4)


<a id="1"></a>
## 1. 两阶段范式 ⭐ / The Two-Stage Paradigm

| | 预训练 Pretraining | 微调 Fine-tuning |
|---|---|---|
| 数据 | 海量**无标注**文本(TB 级网页/书/代码) | 少量**特定**数据(任务/领域/风格) |
| 目标 | 自监督(预测下一个token) | 适配具体任务(可有标注) |
| 成本 | **极高**(百万美元、数千 GPU、数周) | **低**(几张 GPU、几小时) |
| 谁做 | 少数大机构(OpenAI/Google/Meta…) | 几乎人人 |
| 产物 | 通用基础模型(GPT/LLaMA…) | 你的专属模型 |

这套范式本质就是**迁移学习**(呼应 9.13、10.11)：**昂贵的通用能力学一次，便宜地复用到无数下游任务**。它把"训练强大模型"的门槛从"自己从零训"降到"在开源基础模型上微调"——这是开源 LLM 生态繁荣的根基。
This paradigm is essentially **transfer learning** (echoing 9.13, 10.11): **learn expensive general ability once, cheaply reuse it across countless downstream tasks**. It lowers the bar from "train from scratch" to "fine-tune an open base model" — the foundation of the open-LLM ecosystem.


<a id="2"></a>
## 2. 预训练学到了什么 ⭐ / What Pretraining Learns

仅靠"预测下一个 token"这个简单目标，在足够大的数据和模型上，竟能学到惊人丰富的东西(面试常问"为什么 next-token prediction 这么强")：
From the simple objective of "predict the next token," at sufficient data/model scale, models learn surprisingly rich things (interviewers ask "why is next-token prediction so powerful"):
- **语言能力**：语法、拼写、文体、多语言。
  **Language:** grammar, spelling, style, multilingual.
- **世界知识**：事实("巴黎是法国首都")、常识。
  **World knowledge:** facts ("Paris is France's capital"), common sense.
- **隐含技能**：要准确预测下一个词，模型被迫学会**推理、翻译、摘要、甚至算术**(因为这些都出现在训练文本里)。
  **Implicit skills:** to predict the next word well, the model is forced to learn **reasoning, translation, summarization, even arithmetic** (all present in the text).
- **涌现能力(emergent abilities)**：模型够大时，**上下文学习(in-context learning)** 等能力"突然"出现——给几个例子就能做新任务(为 12.11 prompt 铺路)。
  **Emergent abilities:** at scale, abilities like **in-context learning** "suddenly" appear — do a new task from a few examples (setting up 12.11 prompting).

一句话：**预训练把"通用智能"压进了模型权重**；微调只是把这种通用能力"指向"你的具体任务。
In short: **pretraining compresses "general competence" into the weights**; fine-tuning merely "points" that competence at your specific task.


<a id="3"></a>
## 3. 实测：微调 vs 从零 ⭐ / Experiment: Fine-tune vs From-Scratch

我们用字符级 GPT 演示核心价值：**先在简·奥斯汀文本上预训练**(学通用英文)，再到**风格不同的《爱丽丝梦游仙境》**上分别(a)**微调预训练模型** 和 (b)**从零训练**，对比在新文本上的收敛。
We demo the core value with a char-level GPT: **pretrain on Jane Austen** (learn general English), then on the **different-style *Alice in Wonderland*** either (a) **fine-tune the pretrained model** or (b) **train from scratch**, comparing convergence on the new text.


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns, math, time, copy
import torch, torch.nn as nn, torch.nn.functional as F
import nltk; nltk.download("gutenberg", quiet=True); from nltk.corpus import gutenberg
sns.set_theme(style="whitegrid"); torch.manual_seed(0)

pre_text = gutenberg.raw("austen-sense.txt")[:150000].lower()    # 预训练语料(大) / pretrain corpus
ft_text  = gutenberg.raw("carroll-alice.txt")[:30000].lower()    # 微调语料(小, 风格不同) / fine-tune corpus (small)
chars = sorted(set(pre_text + ft_text)); V = len(chars); stoi = {c:i for i,c in enumerate(chars)}
def enc(t): return torch.tensor([stoi[c] for c in t])
CTX = 48
def get_batch(d, B=64):
    ix = torch.randint(len(d)-CTX-1, (B,))
    return torch.stack([d[i:i+CTX] for i in ix]), torch.stack([d[i+1:i+CTX+1] for i in ix])

class MHA(nn.Module):
    def __init__(s,D,H): super().__init__(); s.H=H;s.d=D//H;s.qkv=nn.Linear(D,3*D);s.o=nn.Linear(D,D); s.register_buffer("m",torch.tril(torch.ones(CTX,CTX)))
    def forward(s,x):
        B,T,D=x.shape; qkv=s.qkv(x).reshape(B,T,3,s.H,s.d).permute(2,0,3,1,4); q,k,v=qkv
        sc=(q@k.transpose(-2,-1)/math.sqrt(s.d)).masked_fill(s.m[:T,:T]==0,float("-inf"))
        return s.o((F.softmax(sc,-1)@v).transpose(1,2).reshape(B,T,D))
class Blk(nn.Module):
    def __init__(s,D,H): super().__init__(); s.a=MHA(D,H);s.n1=nn.LayerNorm(D);s.n2=nn.LayerNorm(D);s.ff=nn.Sequential(nn.Linear(D,4*D),nn.GELU(),nn.Linear(4*D,D))
    def forward(s,x): x=x+s.a(s.n1(x)); return x+s.ff(s.n2(x))
class GPT(nn.Module):
    def __init__(s,V,D=96,H=4,L=2): super().__init__(); s.tok=nn.Embedding(V,D);s.pos=nn.Embedding(CTX,D);s.blocks=nn.ModuleList([Blk(D,H) for _ in range(L)]);s.ln=nn.LayerNorm(D);s.head=nn.Linear(D,V)
    def forward(s,x):
        T=x.size(1); h=s.tok(x)+s.pos(torch.arange(T))
        for b in s.blocks: h=b(h)
        return s.head(s.ln(h))

pre = enc(pre_text); ft = enc(ft_text); n=int(0.9*len(ft)); ft_tr, ft_va = ft[:n], ft[n:]
# 1) 预训练 / pretrain on Austen
torch.manual_seed(0); base = GPT(V); opt = torch.optim.AdamW(base.parameters(), 3e-3); t0=time.time()
for _ in range(800):
    x,y = get_batch(pre); opt.zero_grad(); F.cross_entropy(base(x).reshape(-1,V), y.reshape(-1)).backward(); opt.step()
pre_state = copy.deepcopy(base.state_dict())
print(f"预训练完成(在奥斯汀文本上, {time.time()-t0:.0f}s)")

def finetune(from_pretrained, steps=150):
    torch.manual_seed(1); m = GPT(V)
    if from_pretrained: m.load_state_dict(pre_state)     # 从预训练权重开始 / start from pretrained weights
    o = torch.optim.AdamW(m.parameters(), 1e-3); curve=[]
    for s in range(steps):
        x,y = get_batch(ft_tr); o.zero_grad(); F.cross_entropy(m(x).reshape(-1,V), y.reshape(-1)).backward(); o.step()
        if s % 15 == 0:
            with torch.no_grad(): vx,vy=get_batch(ft_va); curve.append(F.cross_entropy(m(vx).reshape(-1,V),vy.reshape(-1)).item())
    return curve

c_pre = finetune(True); c_scratch = finetune(False)
fig, ax = plt.subplots(figsize=(7.5,4))
steps = np.arange(len(c_pre))*15
ax.plot(steps, c_pre, "o-", color="#39c", label="微调预训练模型")
ax.plot(steps, c_scratch, "s-", color="#e67", label="从零训练")
ax.set_xlabel("在新文本上的训练步"); ax.set_ylabel("验证损失(越低越好)"); ax.legend()
ax.set_title("微调 vs 从零: 预训练模型在新数据上收敛更快、更低")
plt.tight_layout(); plt.show()
print(f"微调预训练模型: 最终验证损失 {c_pre[-1]:.2f}")
print(f"从零训练:       最终验证损失 {c_scratch[-1]:.2f}")
print("预训练学到的通用英文能力 → 在新文本(不同风格)上几步就适配; 从零则慢且差(数据少)")


<a id="4"></a>
## 4. 决策指南: 微调/PEFT/Prompt/RAG + 小结 ⭐ / Decision Guide

有了预训练大模型，适配到你的需求有**四条路**(面试常考"怎么选")，从轻到重：
With a pretrained LLM, four ways to adapt to your needs (interviewers ask "how to choose"), light to heavy:
1. **提示工程(Prompting, 12.11)**：不改模型，靠写好 prompt / 给几个例子(few-shot)。**最快最便宜**，适合通用任务、快速试。
   **Prompting:** don't touch weights; craft prompts / few-shot examples. **Fastest, cheapest**; good for general tasks and quick trials.
2. **检索增强(RAG, 12.12)**：不改模型，**外挂知识库**检索相关内容塞进 prompt。适合**需要最新/私有/事实性知识**(减少幻觉)。
   **RAG:** don't touch weights; **attach a knowledge base**, retrieve relevant context into the prompt. Good for **fresh/private/factual** knowledge (reduces hallucination).
3. **参数高效微调(PEFT/LoRA, 12.9)**：只训**很小一部分新参数**(冻结主干)。适合需要**改变模型行为/风格**但算力有限时。
   **PEFT/LoRA:** train only a **tiny set of new parameters** (freeze the backbone). Good to **change behavior/style** with limited compute.
4. **全量微调(Full fine-tuning)**：更新**所有参数**。效果上限最高，但**贵、需要较多数据、且有灾难性遗忘风险**(把通用能力训没了)。
   **Full fine-tuning:** update **all parameters**. Highest ceiling but **expensive, data-hungry, and risks catastrophic forgetting** (losing general ability).

**经验法则**：先试 **Prompt** → 知识问题加 **RAG** → 还不够再 **LoRA 微调** → 资源充足且追求极致才 **全量微调**。
**Rule of thumb:** start with **prompting** → add **RAG** for knowledge → then **LoRA** if needed → **full fine-tuning** only with ample resources for max quality.

> **指令微调(instruction tuning) + RLHF**：把基础模型(只会续写)变成"听话的助手"(ChatGPT)的关键一步——用大量"指令→理想回答"数据微调，再用人类偏好对齐(12.10)。
> **Instruction tuning + RLHF:** the key step turning a base model (just continues text) into a "helpful assistant" (ChatGPT) — fine-tune on "instruction→ideal answer" data, then align with human preferences (12.10).

```
两阶段: 预训练(海量无标注+预测下一token, 极贵, 学通用能力/知识) → 微调(少量特定数据, 便宜, 适配任务)
本质=迁移学习: 贵的通用能力学一次, 便宜复用到下游; 开源LLM生态的根基
预训练学到: 语言+世界知识+隐含技能(推理/翻译/算术)+涌现能力(in-context learning)
实测: 微调预训练模型在新数据上收敛更快更低(vs 从零, 尤其数据少时)
适配四路(轻→重): Prompt → RAG → PEFT/LoRA → 全量微调; 经验法则按此顺序升级
灾难性遗忘: 全量微调可能训没通用能力; 指令微调+RLHF 把基础模型变成对话助手
```

### 💡 面试速查 / Interview cheat-sheet
1. **两阶段**: 预训练(贵/通用/无标注) → 微调(便宜/特定); 本质迁移学习。
   Two stages: pretrain (expensive/general) → fine-tune (cheap/specific); transfer learning.
2. **预训练学什么**: 语言+知识+隐含技能+涌现能力(从next-token目标)。
   Pretraining learns: language+knowledge+implicit skills+emergent abilities.
3. **适配四路**: Prompt < RAG < PEFT/LoRA < 全量微调(成本与效果递增)。
   Four ways: Prompt < RAG < PEFT/LoRA < full fine-tune (rising cost/power).
4. **怎么选**: 先Prompt, 知识问题加RAG, 行为改变用LoRA, 极致才全量。
   How to choose: Prompt first, RAG for knowledge, LoRA for behavior, full only for max.
5. **灾难性遗忘**: 全量微调可能丢通用能力; 指令微调+RLHF→对话助手。
   Catastrophic forgetting: full fine-tune can lose general ability; instruction tuning+RLHF → assistant.

### 下一节 / Next
**12.9 参数高效微调(PEFT/LoRA)**——全量微调一个百亿参数模型要存所有梯度和优化器状态, 极耗显存。**LoRA** 只训练注入的**低秩小矩阵**(冻结原模型), 参数量减少上千倍却效果接近。我们会**从零实现 LoRA**。
**12.9 PEFT/LoRA** — full fine-tuning a billion-parameter model stores all gradients/optimizer states, eating memory. **LoRA** trains only injected **low-rank matrices** (freezing the original), cutting trainable params thousands-fold with near-equal quality. We'll **implement LoRA from scratch**.
